# NB5 — React Web App (Mobile-First)

Mobile-first dashboard with 4 screens:

| Screen | What it shows |
|---|---|
| **Home** | Animated health gauge, RUL countdown, stat cards, alert feed |
| **Trends** | 24h sparklines — HI, RUL, temperature, bounce time |
| **Engineer** | RAG diagnostic report, checklist, KB provenance |
| **Simulator** | Live what-if sliders (physics model in JS) |

**Setup:**
```bash
npm create vite@latest aria-app -- --template react
cd aria-app
# run the file writer cell below, then:
npm install && npm run dev
```

**API wiring:** flip `API_MODE = true` → points at Notebook 4 server.

**ARIA — Relay Intelligence · APOGEE'26 · BITS Pilani · Luminous Power Technologies**

## App.jsx Source

The complete React app. Run the cell below to write it to disk.

In [ ]:
APP_JSX = """/**
 * ARIA — Adaptive Relay Intelligence & Anomaly System
 * Notebook 5: React Web App (Mobile-First)
 *
 * Save as: aria-app/src/App.jsx
 * Run with: npm create vite@latest aria-app -- --template react
 *           cd aria-app && npm install && npm run dev
 *
 * In demo mode (API_MODE=false) all data is mocked — no backend needed.
 * Flip API_MODE=true and set API_BASE to your Notebook 4 server URL.
 *
 * Four screens (bottom-nav):
 *   1. Home       — health gauge + RUL countdown + quick alerts
 *   2. Trends     — 24h sparklines for HI, temp, bounce time
 *   3. Engineer   — RAG diagnostic report + action checklist
 *   4. Simulator  — what-if slider panel
 */

import { useState, useEffect, useCallback } from "react";

// ─── Config ────────────────────────────────────────────────────────────────
const API_MODE  = false;                          // true = live Notebook 4 server
const API_BASE  = "http://localhost:8000";
const DEVICE_ID = "INV-DEMO-001";
const POLL_MS   = 10_000;

// ─── Mock data (used when API_MODE = false) ─────────────────────────────────
const MOCK = {
  health: {
    device_id:    DEVICE_ID,
    health_index: 22.5,
    alert_level:  "ORANGE",
    rul_days:     19,
    rul_ci:       [13, 28],
    is_anomaly:   false,
    temperature_C:38.2,
    open_alerts:  2,
  },
  telemetry: Array.from({ length: 48 }, (_, i) => {
    const hi = Math.max(5, 95 - i * 1.5 + (Math.random() - 0.5) * 4);
    return {
      timestamp:                      new Date(Date.now() - (47-i)*30*60*1000).toISOString(),
      reg_4002_relay_health_index:    parseFloat(hi.toFixed(1)),
      rul_days:                       parseFloat((hi * 1.9).toFixed(1)),
      reg_3019_temp_C:                parseFloat((35 + Math.random()*8).toFixed(1)),
      reg_4003_contact_bounce_ms:     parseFloat((2 + (100-hi)*0.06 + Math.random()*0.3).toFixed(2)),
      alert_level: hi > 60 ? "GREEN" : hi > 30 ? "YELLOW" : hi > 10 ? "ORANGE" : "RED",
    };
  }),
  alerts: [
    { alert_id:"RPT-001", alert_level:"ORANGE", health_index:22.5, rul_days:19,
      message:"Your inverter's switching relay has been working hard handling heavy loads. It's at 22% health with about 19 days of life remaining.",
      fired_at: new Date(Date.now()-3600*1000).toISOString() },
    { alert_id:"RPT-002", alert_level:"YELLOW", health_index:38, rul_days:45,
      message:"Moderate wear detected. Plan a service visit in the next 4–6 weeks.",
      fired_at: new Date(Date.now()-2*24*3600*1000).toISOString() },
  ],
  report: {
    report_id: "RPT-001",
    alert: {
      health_index: 22.5, severity:"HIGH", rul_days:19,
      rul_ci:[13,28], failure_mode:"ARC_EROSION", urgency_days:14,
    },
    customer: {
      message:"Your inverter's switching relay has been working hard handling heavy loads. Think of it like brake pads — they wear down with use. It's at 22% health with about 19 days of life remaining.",
      action:"Schedule service within 14 days",
      plain_rul:"About 19 days remaining (13–28 day range)",
    },
    engineer: {
      failure_mode:"ARC_EROSION",
      actions:[
        "Measure contact resistance (expect >150mΩ)",
        "Check bounce time (expect >4ms)",
        "Review load profile for motor/AC loads",
        "Schedule relay replacement",
        "Log switch count at replacement",
      ],
      service_refs:["SP-001 Relay Replacement"],
      similar_cases:["Case Study: AC Compressor Inrush — Premature Relay Failure"],
    },
    visual_inspection: {
      visual_label:"moderate_erosion",
      severity:"ORANGE",
      description:"Moderate crater pitting. Schedule replacement.",
      hi_estimate:40,
    },
    retrieval: { sources_used:["sys_001","sys_002","ds_002"] },
  },
};

// ─── API layer ──────────────────────────────────────────────────────────────
async function apiFetch(path) {
  if (!API_MODE) return null;
  const r = await fetch(`${API_BASE}${path}`);
  if (!r.ok) throw new Error(`${r.status}`);
  return r.json();
}

async function apiPost(path, body) {
  if (!API_MODE) return null;
  const r = await fetch(`${API_BASE}${path}`, {
    method:"POST", headers:{"Content-Type":"application/json"},
    body: JSON.stringify(body),
  });
  return r.json();
}

// ─── Colour helpers ─────────────────────────────────────────────────────────
const LEVEL_COLOR = {
  GREEN: "#22c55e", YELLOW: "#eab308", ORANGE: "#f97316", RED: "#ef4444",
};
const HI_COLOR = (hi) =>
  hi > 60 ? "#22c55e" : hi > 30 ? "#eab308" : hi > 10 ? "#f97316" : "#ef4444";

// ─── Tiny sparkline ─────────────────────────────────────────────────────────
function Sparkline({ data, color = "#60a5fa", height = 48 }) {
  if (!data?.length) return null;
  const w = 280, h = height;
  const min = Math.min(...data), max = Math.max(...data);
  const range = max - min || 1;
  const pts = data.map((v, i) => {
    const x = (i / (data.length - 1)) * w;
    const y = h - ((v - min) / range) * (h - 4) - 2;
    return `${x.toFixed(1)},${y.toFixed(1)}`;
  }).join(" ");
  return (
    <svg width="100%" viewBox={`0 0 ${w} ${h}`} style={{ display:"block" }}>
      <polyline points={pts} fill="none" stroke={color} strokeWidth="2"
        strokeLinecap="round" strokeLinejoin="round" />
      <polyline points={`0,${h} ${pts} ${w},${h}`}
        fill={color} fillOpacity="0.12" stroke="none" />
    </svg>
  );
}

// ─── Circular gauge ─────────────────────────────────────────────────────────
function HealthGauge({ hi = 0, rul = 0, level = "GREEN" }) {
  const R = 80, C = 2 * Math.PI * R;
  const pct = Math.max(0, Math.min(hi, 100)) / 100;
  const color = HI_COLOR(hi);
  const [anim, setAnim] = useState(0);
  useEffect(() => {
    const t = setTimeout(() => setAnim(pct), 120);
    return () => clearTimeout(t);
  }, [pct]);

  return (
    <div style={{ display:"flex", flexDirection:"column", alignItems:"center", gap:8 }}>
      <svg width={200} height={200} viewBox="0 0 200 200">
        {/* track */}
        <circle cx={100} cy={100} r={R} fill="none" stroke="#1e293b"
          strokeWidth={14} />
        {/* progress */}
        <circle cx={100} cy={100} r={R} fill="none" stroke={color}
          strokeWidth={14} strokeLinecap="round"
          strokeDasharray={C}
          strokeDashoffset={C - anim * C}
          transform="rotate(-90 100 100)"
          style={{ transition:"stroke-dashoffset 1s cubic-bezier(.4,0,.2,1), stroke .5s" }} />
        {/* HI number */}
        <text x={100} y={92} textAnchor="middle" dominantBaseline="middle"
          fill={color} fontSize={36} fontWeight={700}
          fontFamily="'DM Mono', monospace">{Math.round(hi)}</text>
        <text x={100} y={118} textAnchor="middle"
          fill="#64748b" fontSize={12} fontFamily="'DM Sans', sans-serif">HEALTH INDEX</text>
        {/* RUL */}
        <text x={100} y={142} textAnchor="middle"
          fill="#94a3b8" fontSize={11} fontFamily="'DM Sans', sans-serif">{Math.round(rul)} days left</text>
      </svg>
      <span style={{
        background: color + "22", color, border:`1px solid ${color}44`,
        borderRadius:20, padding:"4px 16px", fontSize:12, fontWeight:600,
        fontFamily:"'DM Sans', sans-serif", letterSpacing:1,
      }}>
        {level === "GREEN" ? "● HEALTHY" : level === "YELLOW" ? "● MONITOR"
          : level === "ORANGE" ? "⚠ ATTENTION" : "⚠ CRITICAL"}
      </span>
    </div>
  );
}

// ─── Stat card ──────────────────────────────────────────────────────────────
function StatCard({ label, value, unit, sub, accent }) {
  return (
    <div style={{
      background:"#0f172a", border:"1px solid #1e293b",
      borderRadius:12, padding:"14px 16px", flex:1,
      borderLeft: accent ? `3px solid ${accent}` : undefined,
    }}>
      <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
        letterSpacing:1, marginBottom:4 }}>{label}</div>
      <div style={{ color:"#f1f5f9", fontSize:22, fontWeight:700,
        fontFamily:"'DM Mono',monospace" }}>
        {value}<span style={{ fontSize:12, color:"#64748b", marginLeft:3 }}>{unit}</span>
      </div>
      {sub && <div style={{ color:"#475569", fontSize:10, marginTop:3 }}>{sub}</div>}
    </div>
  );
}

// ─── Alert pill ─────────────────────────────────────────────────────────────
function AlertPill({ alert, onOpen }) {
  const c = LEVEL_COLOR[alert.alert_level];
  const ago = Math.round((Date.now() - new Date(alert.fired_at))/60000);
  return (
    <button onClick={() => onOpen(alert)} style={{
      width:"100%", textAlign:"left", background:"#0f172a",
      border:`1px solid ${c}44`, borderLeft:`3px solid ${c}`,
      borderRadius:10, padding:"12px 14px", cursor:"pointer",
      display:"flex", alignItems:"flex-start", gap:12, marginBottom:8,
    }}>
      <span style={{ color:c, fontSize:18, lineHeight:1 }}>●</span>
      <div style={{ flex:1 }}>
        <div style={{ color:"#f1f5f9", fontSize:13, lineHeight:1.5 }}>
          {alert.message.slice(0,90)}…
        </div>
        <div style={{ color:"#475569", fontSize:10, marginTop:4, fontFamily:"'DM Mono',monospace" }}>
          HI {alert.health_index}% · RUL {alert.rul_days}d · {ago}m ago
        </div>
      </div>
    </button>
  );
}

// ═══════════════════════════════════════════════════════
// SCREEN 1 — HOME
// ═══════════════════════════════════════════════════════
function HomeScreen({ health, alerts, onOpenReport }) {
  if (!health) return <Loader />;
  const { health_index:hi, alert_level:level, rul_days:rul,
          rul_ci, temperature_C:temp, open_alerts } = health;
  return (
    <div style={{ padding:"24px 16px", maxWidth:420, margin:"0 auto" }}>
      {/* Header */}
      <div style={{ marginBottom:28 }}>
        <div style={{ color:"#475569", fontSize:11, fontFamily:"'DM Mono',monospace",
          letterSpacing:2 }}>LUMINOUS · {DEVICE_ID}</div>
        <h1 style={{ color:"#f1f5f9", fontSize:22, fontWeight:700, margin:"4px 0 0",
          fontFamily:"'DM Sans',sans-serif" }}>Relay Health</h1>
      </div>

      {/* Gauge */}
      <div style={{ display:"flex", justifyContent:"center", marginBottom:28 }}>
        <HealthGauge hi={hi} rul={rul} level={level} />
      </div>

      {/* Stats row */}
      <div style={{ display:"flex", gap:10, marginBottom:10 }}>
        <StatCard label="RUL" value={Math.round(rul)} unit="days"
          sub={`90% CI: ${rul_ci[0]}–${rul_ci[1]}d`} accent={HI_COLOR(hi)} />
        <StatCard label="TEMP" value={temp?.toFixed(1) ?? "—"} unit="°C"
          accent={temp > 45 ? "#f97316" : "#22c55e"} />
      </div>
      <div style={{ display:"flex", gap:10, marginBottom:24 }}>
        <StatCard label="OPEN ALERTS" value={open_alerts} unit=""
          accent={open_alerts > 0 ? "#ef4444" : "#22c55e"} />
        <StatCard label="ANOMALY" value={health.is_anomaly ? "YES" : "NO"} unit=""
          accent={health.is_anomaly ? "#ef4444" : "#22c55e"} />
      </div>

      {/* Customer plain-language message */}
      {level !== "GREEN" && alerts[0] && (
        <div style={{
          background:"#f9731611", border:"1px solid #f9731633",
          borderRadius:12, padding:"14px 16px", marginBottom:24,
        }}>
          <div style={{ color:"#fb923c", fontSize:11, fontFamily:"'DM Mono',monospace",
            letterSpacing:1, marginBottom:6 }}>WHAT THIS MEANS</div>
          <div style={{ color:"#e2e8f0", fontSize:13, lineHeight:1.6 }}>
            {alerts[0].message}
          </div>
          <button onClick={() => onOpenReport(alerts[0])} style={{
            marginTop:12, background:"#f97316", color:"#fff", border:"none",
            borderRadius:8, padding:"8px 16px", fontSize:12, fontWeight:600,
            fontFamily:"'DM Sans',sans-serif", cursor:"pointer",
          }}>
            View full diagnosis →
          </button>
        </div>
      )}

      {/* Recent alerts */}
      {alerts.length > 0 && (
        <>
          <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
            letterSpacing:1, marginBottom:10 }}>RECENT ALERTS</div>
          {alerts.slice(0,3).map(a =>
            <AlertPill key={a.alert_id} alert={a} onOpen={onOpenReport} />)}
        </>
      )}
    </div>
  );
}

// ═══════════════════════════════════════════════════════
// SCREEN 2 — TRENDS
// ═══════════════════════════════════════════════════════
function TrendsScreen({ telemetry }) {
  if (!telemetry?.length) return <Loader />;
  const hi     = telemetry.map(r => r.reg_4002_relay_health_index);
  const temp   = telemetry.map(r => r.reg_3019_temp_C);
  const bounce = telemetry.map(r => r.reg_4003_contact_bounce_ms);
  const rul    = telemetry.map(r => r.rul_days);
  const latest = telemetry[telemetry.length-1];

  const charts = [
    { label:"HEALTH INDEX",    data:hi,     unit:"%",  color:"#22c55e",
      val:hi[hi.length-1]?.toFixed(1), note:"Lower = more wear" },
    { label:"RUL ESTIMATE",    data:rul,    unit:"d",  color:"#60a5fa",
      val:rul[rul.length-1]?.toFixed(1), note:"Days of life remaining" },
    { label:"TEMPERATURE",     data:temp,   unit:"°C", color:"#f97316",
      val:temp[temp.length-1]?.toFixed(1), note:">45°C accelerates wear" },
    { label:"BOUNCE TIME",     data:bounce, unit:"ms", color:"#a78bfa",
      val:bounce[bounce.length-1]?.toFixed(2), note:">4ms = spring fatigue" },
  ];

  return (
    <div style={{ padding:"24px 16px", maxWidth:420, margin:"0 auto" }}>
      <div style={{ color:"#475569", fontSize:11, fontFamily:"'DM Mono',monospace",
        letterSpacing:2, marginBottom:4 }}>LAST 24 HOURS</div>
      <h1 style={{ color:"#f1f5f9", fontSize:22, fontWeight:700, margin:"0 0 24px",
        fontFamily:"'DM Sans',sans-serif" }}>Trends</h1>

      {charts.map(ch => (
        <div key={ch.label} style={{
          background:"#0f172a", border:"1px solid #1e293b",
          borderRadius:14, padding:"16px", marginBottom:14,
        }}>
          <div style={{ display:"flex", justifyContent:"space-between",
            alignItems:"flex-start", marginBottom:10 }}>
            <div>
              <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
                letterSpacing:1 }}>{ch.label}</div>
              <div style={{ color:ch.color, fontSize:24, fontWeight:700,
                fontFamily:"'DM Mono',monospace" }}>
                {ch.val}<span style={{ fontSize:12, color:"#64748b" }}>{ch.unit}</span>
              </div>
            </div>
            <div style={{ color:"#334155", fontSize:10, textAlign:"right",
              maxWidth:100, lineHeight:1.4 }}>{ch.note}</div>
          </div>
          <Sparkline data={ch.data} color={ch.color} height={52} />
        </div>
      ))}

      {/* Physics explainer */}
      <div style={{
        background:"#1e293b44", border:"1px solid #1e293b",
        borderRadius:12, padding:"14px 16px", marginTop:8,
      }}>
        <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
          letterSpacing:1, marginBottom:8 }}>HOW WEAR IS CALCULATED</div>
        <div style={{ color:"#94a3b8", fontSize:12, lineHeight:1.7 }}>
          Each relay switch causes arc damage proportional to <span style={{color:"#60a5fa"}}>I²</span>.
          An AC compressor start at 15A causes <span style={{color:"#f97316"}}>225×</span> more
          wear than a 1A switch. Temperature above 25°C multiplies wear via the
          Arrhenius factor. ARIA tracks both continuously.
        </div>
      </div>
    </div>
  );
}

// ═══════════════════════════════════════════════════════
// SCREEN 3 — ENGINEER (RAG Diagnostic)
// ═══════════════════════════════════════════════════════
const MODE_COLORS = {
  ARC_EROSION:"#f97316", THERMAL_DEGRADATION:"#ef4444",
  CONTACT_OXIDATION:"#a78bfa", CONTACT_WELDING:"#f43f5e",
};
const MODE_ICON = {
  ARC_EROSION:"⚡", THERMAL_DEGRADATION:"🌡", CONTACT_OXIDATION:"💧", CONTACT_WELDING:"🔗",
};

function EngineerScreen({ report }) {
  const [checked, setChecked] = useState({});
  if (!report) return (
    <div style={{ padding:32, textAlign:"center", color:"#475569" }}>
      No active diagnostic report.<br />
      <span style={{ fontSize:12 }}>Alerts appear when HI drops below 30%.</span>
    </div>
  );
  const { alert:a, engineer:eng, visual_inspection:vi, retrieval } = report;
  const modeColor = MODE_COLORS[a.failure_mode] || "#60a5fa";

  const toggle = (i) => setChecked(prev => ({...prev, [i]: !prev[i]}));
  const done   = Object.values(checked).filter(Boolean).length;

  return (
    <div style={{ padding:"24px 16px", maxWidth:420, margin:"0 auto" }}>
      <div style={{ color:"#475569", fontSize:11, fontFamily:"'DM Mono',monospace",
        letterSpacing:2, marginBottom:4 }}>SERVICE ENGINEER VIEW</div>
      <h1 style={{ color:"#f1f5f9", fontSize:22, fontWeight:700, margin:"0 0 20px",
        fontFamily:"'DM Sans',sans-serif" }}>Diagnosis</h1>

      {/* Failure mode badge */}
      <div style={{
        background: modeColor+"1a", border:`1px solid ${modeColor}44`,
        borderRadius:12, padding:"14px 16px", marginBottom:16,
        display:"flex", alignItems:"center", gap:12,
      }}>
        <span style={{ fontSize:28 }}>{MODE_ICON[a.failure_mode]}</span>
        <div>
          <div style={{ color:modeColor, fontSize:11, fontFamily:"'DM Mono',monospace",
            letterSpacing:1 }}>PRIMARY FAILURE MODE</div>
          <div style={{ color:"#f1f5f9", fontSize:16, fontWeight:700,
            fontFamily:"'DM Sans',sans-serif", marginTop:2 }}>
            {a.failure_mode.replace(/_/g," ")}
          </div>
        </div>
        <div style={{ marginLeft:"auto", textAlign:"right" }}>
          <div style={{ color:modeColor, fontSize:22, fontWeight:700,
            fontFamily:"'DM Mono',monospace" }}>{a.health_index}%</div>
          <div style={{ color:"#64748b", fontSize:10 }}>HI</div>
        </div>
      </div>

      {/* RUL with CI */}
      <div style={{ background:"#0f172a", border:"1px solid #1e293b",
        borderRadius:12, padding:"14px 16px", marginBottom:16 }}>
        <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
          letterSpacing:1, marginBottom:6 }}>REMAINING USEFUL LIFE</div>
        <div style={{ display:"flex", alignItems:"baseline", gap:8 }}>
          <span style={{ color:"#f1f5f9", fontSize:28, fontWeight:700,
            fontFamily:"'DM Mono',monospace" }}>{a.rul_days}</span>
          <span style={{ color:"#64748b", fontSize:13 }}>days</span>
          <span style={{ color:"#334155", fontSize:12, marginLeft:8 }}>
            90% CI: [{a.rul_ci[0]}–{a.rul_ci[1]}d]
          </span>
        </div>
        <div style={{ color:"#f97316", fontSize:12, marginTop:6 }}>
          ⚠ Schedule service within {a.urgency_days} days
        </div>
      </div>

      {/* Visual inspection */}
      {vi && (
        <div style={{ background:"#0f172a", border:"1px solid #1e293b",
          borderRadius:12, padding:"14px 16px", marginBottom:16 }}>
          <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
            letterSpacing:1, marginBottom:8 }}>VISUAL INSPECTION (PHOTO)</div>
          <div style={{ display:"flex", alignItems:"center", gap:12 }}>
            {/* Simulated contact image */}
            <div style={{ width:64, height:64, borderRadius:8, background:"#1e293b",
              display:"flex", alignItems:"center", justifyContent:"center",
              border:"1px solid #334155", fontSize:28 }}>🔬</div>
            <div>
              <div style={{ color:LEVEL_COLOR[vi.severity]||"#f97316", fontSize:13,
                fontWeight:600 }}>{vi.visual_label?.replace(/_/g," ")}</div>
              <div style={{ color:"#94a3b8", fontSize:12, marginTop:2 }}>{vi.description}</div>
              <div style={{ color:"#475569", fontSize:10, marginTop:4,
                fontFamily:"'DM Mono',monospace" }}>HI estimate from photo: {vi.hi_estimate}%</div>
            </div>
          </div>
        </div>
      )}

      {/* Action checklist */}
      <div style={{ background:"#0f172a", border:"1px solid #1e293b",
        borderRadius:12, padding:"14px 16px", marginBottom:16 }}>
        <div style={{ display:"flex", justifyContent:"space-between",
          alignItems:"center", marginBottom:12 }}>
          <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
            letterSpacing:1 }}>ACTION CHECKLIST</div>
          <div style={{ color:"#22c55e", fontSize:10, fontFamily:"'DM Mono',monospace" }}>
            {done}/{eng.actions.length} DONE
          </div>
        </div>
        {eng.actions.map((step, i) => (
          <button key={i} onClick={() => toggle(i)} style={{
            width:"100%", textAlign:"left", background:"transparent",
            border:"none", cursor:"pointer",
            display:"flex", alignItems:"flex-start", gap:10,
            padding:"8px 0", borderBottom: i < eng.actions.length-1 ? "1px solid #1e293b" : "none",
          }}>
            <span style={{
              width:20, height:20, borderRadius:5, flexShrink:0, marginTop:1,
              border: checked[i] ? "none" : "1px solid #334155",
              background: checked[i] ? "#22c55e" : "transparent",
              display:"flex", alignItems:"center", justifyContent:"center",
              color:"#fff", fontSize:11,
            }}>{checked[i] ? "✓" : ""}</span>
            <span style={{ color: checked[i] ? "#475569" : "#cbd5e1", fontSize:13,
              textDecoration: checked[i] ? "line-through" : "none", lineHeight:1.5 }}>
              {step}
            </span>
          </button>
        ))}
      </div>

      {/* KB sources (RAG provenance) */}
      <div style={{ background:"#0f172a", border:"1px solid #1e293b",
        borderRadius:12, padding:"14px 16px", marginBottom:16 }}>
        <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
          letterSpacing:1, marginBottom:8 }}>RAG KNOWLEDGE SOURCES</div>
        {retrieval.sources_used.map(src => (
          <div key={src} style={{ color:"#60a5fa", fontSize:12,
            fontFamily:"'DM Mono',monospace", marginBottom:4 }}>· {src}</div>
        ))}
        {eng.similar_cases?.map(c => (
          <div key={c} style={{ color:"#94a3b8", fontSize:11, marginTop:4,
            lineHeight:1.5, paddingLeft:8, borderLeft:"2px solid #1e3a5f" }}>
            {c}
          </div>
        ))}
      </div>

      {/* Service refs */}
      <div style={{ display:"flex", gap:8 }}>
        {eng.service_refs?.map(ref => (
          <div key={ref} style={{ background:"#1e293b", borderRadius:8,
            padding:"8px 12px", fontSize:11, color:"#94a3b8",
            fontFamily:"'DM Mono',monospace", flex:1, textAlign:"center" }}>
            {ref}
          </div>
        ))}
      </div>
    </div>
  );
}

// ═══════════════════════════════════════════════════════
// SCREEN 4 — SIMULATOR
// ═══════════════════════════════════════════════════════
function SimulatorScreen({ health }) {
  const [extraKw,    setExtraKw]    = useState(0);
  const [extraAC,    setExtraAC]    = useState(0);
  const [ambientT,   setAmbientT]   = useState(35);
  const [result,     setResult]     = useState(null);

  const baseHI = health?.health_index ?? 70;

  const simulate = useCallback(async () => {
    if (API_MODE) {
      const r = await apiPost("/simulate/whatif", {
        device_id: DEVICE_ID, extra_load_kw: extraKw,
        extra_ac_starts_day: extraAC, ambient_temp_C: ambientT,
      });
      setResult(r);
    } else {
      // Local physics calculation (mirrors Notebook 4)
      const k = 1.2e-9;
      const arrh = (T) => Math.exp((0.7 / 8.617e-5) * (1/298.15 - 1/(T+273.15)));
      const maxW  = 100 * k * 100 * 12 * 2.0 * 1e5;
      const remW  = maxW * (baseHI / 100);
      const bdw   = 4 * k * 36 * 12 * 2.5 * arrh(30);
      const edw   = (extraAC * 2 * k * 225 * 12 * 3.0 * arrh(ambientT))
                  + (extraKw * k * 100 * 12 * 2.0 * arrh(ambientT));
      const cur   = bdw > 0 ? remW / bdw : 9999;
      const next  = (bdw+edw) > 0 ? remW / (bdw+edw) : 9999;
      const delta = 100 * (1 - next / Math.max(cur, 1));
      setResult({
        current_rul_days:   parseFloat(cur.toFixed(1)),
        projected_rul_days: parseFloat(next.toFixed(1)),
        rul_reduction_pct:  parseFloat(delta.toFixed(1)),
        recommendation: `Adding ${extraKw}kW load and ${extraAC} AC starts/day at ${ambientT}°C ` +
          `reduces relay life by ${delta.toFixed(0)}% (${cur.toFixed(0)}d → ${next.toFixed(0)}d). ` +
          (next < 14 ? "Immediate service recommended." :
           next < 60 ? "Plan proactive replacement." : "Impact is manageable."),
      });
    }
  }, [extraKw, extraAC, ambientT, baseHI]);

  useEffect(() => { simulate(); }, [extraKw, extraAC, ambientT]);

  const Slider = ({ label, value, onChange, min, max, step=1, unit }) => (
    <div style={{ marginBottom:20 }}>
      <div style={{ display:"flex", justifyContent:"space-between",
        marginBottom:8, alignItems:"center" }}>
        <span style={{ color:"#94a3b8", fontSize:13,
          fontFamily:"'DM Sans',sans-serif" }}>{label}</span>
        <span style={{ color:"#f1f5f9", fontSize:15, fontWeight:700,
          fontFamily:"'DM Mono',monospace" }}>{value}{unit}</span>
      </div>
      <input type="range" min={min} max={max} step={step} value={value}
        onChange={e => onChange(Number(e.target.value))}
        style={{ width:"100%", accentColor:"#60a5fa" }} />
      <div style={{ display:"flex", justifyContent:"space-between",
        color:"#334155", fontSize:10, marginTop:4 }}>
        <span>{min}{unit}</span><span>{max}{unit}</span>
      </div>
    </div>
  );

  const reductionColor = !result ? "#64748b"
    : result.rul_reduction_pct > 70 ? "#ef4444"
    : result.rul_reduction_pct > 40 ? "#f97316"
    : result.rul_reduction_pct > 10 ? "#eab308" : "#22c55e";

  return (
    <div style={{ padding:"24px 16px", maxWidth:420, margin:"0 auto" }}>
      <div style={{ color:"#475569", fontSize:11, fontFamily:"'DM Mono',monospace",
        letterSpacing:2, marginBottom:4 }}>PHYSICS SIMULATOR</div>
      <h1 style={{ color:"#f1f5f9", fontSize:22, fontWeight:700, margin:"0 0 8px",
        fontFamily:"'DM Sans',sans-serif" }}>What-If</h1>
      <p style={{ color:"#64748b", fontSize:13, margin:"0 0 24px", lineHeight:1.6 }}>
        Adjust load conditions and see how they affect relay remaining life.
      </p>

      <div style={{ background:"#0f172a", border:"1px solid #1e293b",
        borderRadius:14, padding:"20px 18px", marginBottom:16 }}>
        <Slider label="Extra load"       value={extraKw}  onChange={setExtraKw}
          min={0} max={5} step={0.5} unit="kW" />
        <Slider label="AC starts per day" value={extraAC}  onChange={setExtraAC}
          min={0} max={20} unit="/day" />
        <Slider label="Ambient temp"     value={ambientT} onChange={setAmbientT}
          min={20} max={55} unit="°C" />
      </div>

      {/* Result card */}
      {result && (
        <div style={{
          background:"#0f172a", border:`1px solid ${reductionColor}44`,
          borderRadius:14, padding:"20px 18px",
        }}>
          <div style={{ color:"#475569", fontSize:10, fontFamily:"'DM Mono',monospace",
            letterSpacing:1, marginBottom:16 }}>PROJECTED IMPACT</div>

          <div style={{ display:"flex", gap:12, marginBottom:16 }}>
            <div style={{ flex:1, textAlign:"center" }}>
              <div style={{ color:"#64748b", fontSize:10,
                fontFamily:"'DM Mono',monospace" }}>CURRENT RUL</div>
              <div style={{ color:"#f1f5f9", fontSize:26, fontWeight:700,
                fontFamily:"'DM Mono',monospace" }}>
                {Math.round(result.current_rul_days)}<span style={{fontSize:12,color:"#64748b"}}>d</span>
              </div>
            </div>
            <div style={{ display:"flex", alignItems:"center", color:"#334155",
              fontSize:20 }}>→</div>
            <div style={{ flex:1, textAlign:"center" }}>
              <div style={{ color:"#64748b", fontSize:10,
                fontFamily:"'DM Mono',monospace" }}>PROJECTED RUL</div>
              <div style={{ color:reductionColor, fontSize:26, fontWeight:700,
                fontFamily:"'DM Mono',monospace" }}>
                {Math.round(result.projected_rul_days)}<span style={{fontSize:12,color:"#64748b"}}>d</span>
              </div>
            </div>
          </div>

          {/* Reduction bar */}
          <div style={{ marginBottom:14 }}>
            <div style={{ display:"flex", justifyContent:"space-between",
              marginBottom:6 }}>
              <span style={{ color:"#64748b", fontSize:11 }}>Life reduction</span>
              <span style={{ color:reductionColor, fontWeight:700, fontSize:13,
                fontFamily:"'DM Mono',monospace" }}>
                {result.rul_reduction_pct.toFixed(1)}%
              </span>
            </div>
            <div style={{ background:"#1e293b", borderRadius:4, height:6 }}>
              <div style={{
                background:reductionColor, borderRadius:4, height:6,
                width:`${Math.min(result.rul_reduction_pct, 100)}%`,
                transition:"width .5s cubic-bezier(.4,0,.2,1)",
              }} />
            </div>
          </div>

          <div style={{ color:"#94a3b8", fontSize:12, lineHeight:1.6,
            borderTop:"1px solid #1e293b", paddingTop:12 }}>
            {result.recommendation}
          </div>

          {/* Arrhenius insight */}
          {ambientT > 40 && (
            <div style={{ marginTop:12, background:"#ef444411",
              border:"1px solid #ef444433", borderRadius:8,
              padding:"10px 12px", fontSize:11, color:"#fca5a5", lineHeight:1.6 }}>
              🌡 At {ambientT}°C, thermal stress multiplies wear by{" "}
              <strong>{Math.exp(0.7/8.617e-5*(1/298.15-1/(ambientT+273.15))).toFixed(1)}×</strong>{" "}
              vs 25°C reference (Arrhenius model).
            </div>
          )}
        </div>
      )}
    </div>
  );
}

// ─── Loader ─────────────────────────────────────────────────────────────────
function Loader() {
  return (
    <div style={{ display:"flex", justifyContent:"center",
      alignItems:"center", height:200, color:"#334155",
      fontFamily:"'DM Mono',monospace", fontSize:13 }}>
      loading…
    </div>
  );
}

// ─── Bottom navigation ───────────────────────────────────────────────────────
const NAV = [
  { id:"home",      label:"Home",       icon:"⬡" },
  { id:"trends",    label:"Trends",     icon:"↗" },
  { id:"engineer",  label:"Engineer",   icon:"🔧" },
  { id:"simulator", label:"Simulator",  icon:"◈" },
];

function BottomNav({ active, onChange, alertCount }) {
  return (
    <nav style={{
      position:"fixed", bottom:0, left:0, right:0,
      background:"#0a0f1e", borderTop:"1px solid #1e293b",
      display:"flex", zIndex:100,
    }}>
      {NAV.map(n => (
        <button key={n.id} onClick={() => onChange(n.id)} style={{
          flex:1, background:"transparent", border:"none",
          cursor:"pointer", padding:"10px 0 14px",
          display:"flex", flexDirection:"column", alignItems:"center", gap:4,
          position:"relative",
        }}>
          <span style={{ fontSize:18, filter: active===n.id ? "none" : "grayscale(1) opacity(.5)" }}>
            {n.icon}
          </span>
          <span style={{
            fontSize:9, fontFamily:"'DM Mono',monospace", letterSpacing:0.5,
            color: active===n.id ? "#f1f5f9" : "#334155",
          }}>{n.label.toUpperCase()}</span>
          {n.id==="home" && alertCount > 0 && (
            <span style={{
              position:"absolute", top:6, right:"calc(50% - 18px)",
              background:"#ef4444", borderRadius:10,
              width:16, height:16, fontSize:9, color:"#fff",
              display:"flex", alignItems:"center", justifyContent:"center",
              fontFamily:"'DM Mono',monospace",
            }}>{alertCount}</span>
          )}
          {active===n.id && (
            <span style={{
              position:"absolute", top:0, left:"20%", right:"20%",
              height:2, background:"#60a5fa", borderRadius:1,
            }} />
          )}
        </button>
      ))}
    </nav>
  );
}

// ═══════════════════════════════════════════════════════
// ROOT APP
// ═══════════════════════════════════════════════════════
export default function App() {
  const [screen,    setScreen]    = useState("home");
  const [health,    setHealth]    = useState(MOCK.health);
  const [telemetry, setTelemetry] = useState(MOCK.telemetry);
  const [alerts,    setAlerts]    = useState(MOCK.alerts);
  const [report,    setReport]    = useState(MOCK.report);
  const [modalAlert,setModalAlert]= useState(null);

  // Live polling (when API_MODE = true)
  useEffect(() => {
    if (!API_MODE) return;
    const refresh = async () => {
      try {
        const [h, t, a] = await Promise.all([
          apiFetch(`/devices/${DEVICE_ID}/health`),
          apiFetch(`/devices/${DEVICE_ID}/telemetry?n=48`),
          apiFetch(`/devices/${DEVICE_ID}/alerts`),
        ]);
        if (h) setHealth(h);
        if (t?.samples) setTelemetry(t.samples);
        if (a?.alerts)  setAlerts(a.alerts);
        // Fetch latest report if there are alerts
        if (a?.alerts?.length) {
          const rid = a.alerts[0].alert_id;
          const r   = await apiFetch(`/reports/${rid}`);
          if (r) setReport(r);
        }
      } catch(e) { console.error("Poll error:", e); }
    };
    refresh();
    const interval = setInterval(refresh, POLL_MS);
    return () => clearInterval(interval);
  }, []);

  const openReport = (alert) => {
    setModalAlert(alert);
    setScreen("engineer");
  };

  return (
    <div style={{
      minHeight:"100vh", background:"#060d1a",
      color:"#f1f5f9", fontFamily:"'DM Sans', sans-serif",
      paddingBottom:70,
    }}>
      {/* Google Fonts */}
      <style>{`
        @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;600;700&family=DM+Mono:wght@400;500&display=swap');
        * { box-sizing: border-box; margin: 0; padding: 0; }
        input[type=range] { height: 4px; }
        ::-webkit-scrollbar { width: 4px; }
        ::-webkit-scrollbar-track { background: #0a0f1e; }
        ::-webkit-scrollbar-thumb { background: #1e293b; border-radius: 2px; }
      `}</style>

      {/* Status bar shimmer (mobile feel) */}
      <div style={{
        height:44, background:"#060d1a",
        display:"flex", alignItems:"center", padding:"0 16px",
        justifyContent:"space-between",
      }}>
        <span style={{ color:"#1e3a5f", fontSize:12,
          fontFamily:"'DM Mono',monospace" }}>ARIA v1.0</span>
        <span style={{
          color: health?.alert_level === "GREEN" ? "#22c55e"
               : health?.alert_level === "YELLOW" ? "#eab308"
               : health?.alert_level === "ORANGE" ? "#f97316" : "#ef4444",
          fontSize:11, fontFamily:"'DM Mono',monospace",
        }}>● {health?.alert_level ?? "—"}</span>
      </div>

      {/* Screens */}
      {screen === "home"      && <HomeScreen health={health} alerts={alerts} onOpenReport={openReport} />}
      {screen === "trends"    && <TrendsScreen telemetry={telemetry} />}
      {screen === "engineer"  && <EngineerScreen report={report} />}
      {screen === "simulator" && <SimulatorScreen health={health} />}

      <BottomNav active={screen} onChange={setScreen}
        alertCount={health?.open_alerts ?? 0} />
    </div>
  );
}
"""
print(f"App.jsx: {len(APP_JSX)} chars, {len(APP_JSX.splitlines())} lines")

## Write App.jsx to Disk

In [ ]:
import os
os.makedirs("aria-app/src", exist_ok=True)
with open("aria-app/src/App.jsx","w") as f: f.write(APP_JSX)
print("Written → aria-app/src/App.jsx")
print("Now run: cd aria-app && npm install && npm run dev")

## Vite Entry Files

Create these two minimal files to complete the Vite setup:

In [ ]:
index_html = """<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>ARIA — Relay Intelligence</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>"""

main_jsx = """import React from 'react'
import ReactDOM from 'react-dom/client'
import App from './App.jsx'
ReactDOM.createRoot(document.getElementById('root')).render(<App />)"""

with open("aria-app/index.html","w") as f: f.write(index_html)
with open("aria-app/src/main.jsx","w") as f: f.write(main_jsx)
print("index.html and main.jsx written")

## API Endpoints Reference

| Screen | Endpoint | Method |
|---|---|---|
| Home | `/devices/{id}/health` | GET |
| Trends | `/devices/{id}/telemetry?n=48` | GET |
| Alerts | `/devices/{id}/alerts` | GET |
| Engineer | `/reports/{report_id}` | GET |
| On-demand report | `/devices/{id}/report` | POST |
| Simulator | `/simulate/whatif` | POST |
| Fleet dashboard | `/fleet/overview` | GET |

## Production Build

```bash
npm run build      # creates dist/
npx vercel ./dist  # deploy free on Vercel
```